In [1]:
import os, json, pickle, math, random
import os
import numpy as np
import torch
import torch.nn as nn
import random
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from functions import DoomDataset
from functions import events_to_midi

In [2]:
STEPS_PER_BEAT = 16

CONTEXT = 1024
STRIDE  = 64
MAX_TRANSPOSE = 12
WEIGHT_DECAY = 0.05

VAL_RATIO = 0.1

BATCH_SIZE = 32
D_MODEL    = 256
N_HEADS    = 8
N_LAYERS   = 4
D_FF       = 1024
DROPOUT    = 0.3     # ważne przy małych danych
LR         = 3e-4
EPOCHS     = 60
PATIENCE   = 10
SAVE_EVERY = 5

PROC_DIR = '../data/processed/'
MODELS_DIR = '../models/'
OUTPUT_DIR = '../output/'

os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA GeForce RTX 4070


In [4]:
with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'rb') as f:
    data = pickle.load(f)

with open(os.path.join(PROC_DIR, 'vocab.json')) as f:
    token2id = json.load(f)
id2token = {y: x for x,y in token2id.items()}

sequences = list(data.values())
names = list(data.keys())
VOCAB_SIZE = len(token2id)
PAD_ID = token2id['PAD']
print(f'Sequences: {len(sequences)} | Vocab: {VOCAB_SIZE} | Tokens: {sum(len(s) for s in sequences):,}')

Sequences: 42 | Vocab: 579 | Tokens: 382,035


In [5]:
class TokenEmbedding(nn.Module):
    def __init__(self, d_model, n_tokens):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(num_embeddings = n_tokens, embedding_dim = d_model)
    
    def forward(self, x):
        return self.embedding_layer(x)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        positional_encoding = np.zeros((max_len, d_model))
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                positional_encoding[pos, i] = np.sin(pos / (10000 ** (2*i / d_model)))
                if i+1 < d_model:
                    positional_encoding[pos, i+1] = np.cos(pos / (10000 ** (2*i / d_model)))
        positional_encoding = torch.from_numpy(positional_encoding).float()
        self.register_buffer('pe', positional_encoding)
        
    def forward(self, x):
        return x + self.pe[:x.size(1), :]
        

class MaskedSelfAttention(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.head_dim = head_dim
        self.q = nn.Linear(d_model, head_dim)
        self.k = nn.Linear(d_model, head_dim)
        self.v = nn.Linear(d_model, head_dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, casual_mask):
        q, k, v = self.q(x), self.k(x), self.v(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~casual_mask, -1e4)
        weights = self.softmax(scores)
        return torch.bmm(weights, v)


class MaskedMultiHeadedSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.head_dim = d_model//n_heads
        self.heads = nn.ModuleList([MaskedSelfAttention(d_model, self.head_dim) for _ in range(n_heads)])
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, casual_mask):
        outs = [h(x, casual_mask) for h in self.heads]
        return self.out(torch.cat(outs, dim=2))


class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.l1 = nn.Linear(d_model, ff_dim)
        self.l2 = nn.Linear(ff_dim, d_model)

    def forward(self, x):
        return self.l2(torch.relu(self.l1(x)))


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout):
        super().__init__()
        self.mha = MaskedMultiHeadedSelfAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, ff_dim)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, casual_mask):
        x = x + self.mha(self.ln1(x), casual_mask)
        ff = self.ff(self.ln2(x))
        if self.training:
            ff = self.dropout(ff)
        return x + ff


class DecoderStack(nn.Module):
    def __init__(self, d_model, n_layers, n_heads, ff_dim, dropout):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, ff_dim, dropout) for _ in range(n_layers)])

    def forward(self, x, casual_mask):
        for layer in self.layers:
            x = layer(x, casual_mask)
        return x


class DoomTransformer(nn.Module):
    def __init__(self, n_tokens, max_len=512, d_model=256, n_layers=4, n_heads=4, ff_dim=1024, dropout=0.1):
        super().__init__()
        self.n_tokens = n_tokens
        self.max_len = max_len
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.ff_dim = ff_dim
        self.dropout = dropout

        self.token_embedding = TokenEmbedding(d_model, n_tokens)
        self.positional_encoding = PositionalEncoding(d_model, max_len)
        self.layer_normalization = nn.LayerNorm(d_model)
        self.decoder = DecoderStack(d_model, n_layers, n_heads, ff_dim, dropout)
        self.lm_head = nn.Linear(d_model, n_tokens)

    def forward(self, x):
        s = x.size(1)
        casual_mask = torch.tril(torch.ones(s, s, device=x.device, dtype=torch.bool))
        x = self.token_embedding(x)
        x = self.positional_encoding(x)
        x = self.layer_normalization(x)
        x = self.decoder(x, casual_mask)
        return self.lm_head(x)

    def save_checkpoint(self, path):
        print(f'Saving checkpoint {path}')
        torch.save({
            'number_of_tokens': self.n_tokens,
            'max_sequence_length': self.max_len,
            'embedding_dimension': self.d_model,
            'number_of_layers': self.n_layers,
            'number_of_heads': self.n_heads,
            'feed_forward_dimension': self.ff_dim,
            'dropout_rate': self.dropout,
            'model_state_dict': self.state_dict()
        }, path)

    @staticmethod
    def load_checkpoint(path) -> 'DoomTransformer':
        checkpoint = torch.load(path)
        model = DoomTransformer(
            checkpoint['number_of_tokens'],
            checkpoint['max_sequence_length'],
            checkpoint['embedding_dimension'], 
            checkpoint['number_of_layers'], 
            checkpoint['number_of_heads'],
            checkpoint['feed_forward_dimension'],
            checkpoint['dropout_rate']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        return model.to(device)

In [6]:
random.seed(67)
val_count = math.floor(len(names) * VAL_RATIO)
shuffled = names.copy()
random.shuffle(shuffled)
val_names = set(shuffled[:val_count])

train_seqs = [s for n, s in zip(names, sequences) if n not in val_names]
train_len = sum([len(x) for x in train_seqs])
val_seqs = [s for n, s in zip(names, sequences) if n in val_names]
val_len = sum([len(x) for x in val_seqs])

train_ds = DoomDataset(train_seqs, context=CONTEXT, stride=STRIDE, augment=True, max_transpose=MAX_TRANSPOSE)
val_ds = DoomDataset(val_seqs, context=CONTEXT, stride=STRIDE, augment=False, max_transpose=MAX_TRANSPOSE)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f"Walidation ({len(val_names)}): {sorted(val_names)}")
print(f"Train: {len(train_seqs)} ({train_len}) | Val: {len(val_seqs)} ({val_len})")
print(f"Walidation = {val_len/(train_len+val_len):.2%}% of dataset")

Walidation (4): ['d_dead.mid', 'd_e2m1.mid', 'd_e2m2.mid', 'd_victor.mid']
Train: 38 (350558) | Val: 4 (31477)
Walidation = 8.24%% of dataset


In [ ]:
model = DoomTransformer(
    n_tokens=VOCAB_SIZE,
    max_len=CONTEXT,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    ff_dim=D_FF,
    dropout=DROPOUT
).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params/1e6:.2f}M')

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler('cuda')

print(f'Start loss = ln(VOCAB) = {math.log(VOCAB_SIZE):.2f}')
best_val, epochs_since_best = float('inf'), 0

for epoch in range(EPOCHS):
    model.train()
    tr = []
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            logits = model(x)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tr.append(loss.item())

    model.eval()
    vl = []
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            with autocast('cuda'):
                logits = model(x)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
            vl.append(loss.item())

    tr_loss = sum(tr)/len(tr)
    val_loss = sum(vl)/len(vl)

    if epoch % SAVE_EVERY == 0:
        model.save_checkpoint(f'../models/transformer_ep{epoch:03d}.pt')
    
    flag = ''
    if val_loss < best_val:
        best_val = val_loss
        epochs_since_best = 0
        model.save_checkpoint(os.path.join(MODELS_DIR, 'transformer_best.pt'))
        flag = 'BEST'
    else:
        epochs_since_best += 1
    print(f'EPOCH {epoch:3d} | train loss {tr_loss:.3f} | val loss {val_loss:.3f} {flag}')

    if epochs_since_best >= PATIENCE:
        print(f'Early stop! No improvement since {PATIENCE} epochs')
        break

Parameters: 3.46M
Start loss = ln(VOCAB) = 6.36


In [ ]:
@torch.no_grad()
def generate(model, max_new_tokens=2000, temperature=1.0, top_k=40, seed_ids=None):
    model.eval()
    bos, eos = token2id['BOS'], token2id['EOS']
    ids = [bos] if seed_ids is None else list(seed_ids)
    generated = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        context = generated[:, -CONTEXT:]
        logits = model(context)[:, -1, :] / temperature
        if top_k is not None:
            kth = torch.topk(logits, top_k).values[:, -1, None]
            logits[logits < kth] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_id], dim=1)
        if next_id.item() == eos:
            break
    return generated[0].tolist()

def ids_to_midi_file(ids, path, bpm=95):
    events = [id2token[i] for i in ids]
    midi = events_to_midi(events, STEPS_PER_BEAT, bpm=bpm)
    midi.write(path)
    n_notes = sum(len(inst.notes) for inst in midi.instruments)
    print(f"{path}  |  {len(ids)} tok  |  {n_notes} nut  |  {midi.get_end_time():.1f}s")

In [ ]:
transformer = DoomTransformer.load_checkpoint('../models/transformer_best.pt')
ids = generate(transformer, max_new_tokens=6000, temperature=0.8, top_k=60)
ids_to_midi_file(ids, os.path.join(OUTPUT_DIR, f'transformer_best_temp0.8.mid'))

transformer = DoomTransformer.load_checkpoint('../models/transformer_ep010.pt')
ids = generate(transformer, max_new_tokens=6000, temperature=0.8, top_k=60)
ids_to_midi_file(ids, os.path.join(OUTPUT_DIR, f'transformer_ep010_temp0.8.mid'))

In [ ]:
transformer = DoomTransformer.load_checkpoint('../models/transformer_best.pt')
ids = generate(transformer, max_new_tokens=2000, temperature=0.8, top_k=60,
               seed_ids=sequences[9][:100])
ids_to_midi_file(ids, os.path.join(OUTPUT_DIR, 'gen_seeded.mid'))